In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys
sys.dont_write_bytecode = True
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(variable, '1')
NOTEBOOK_NAME = '07_00_mri_preprocessing_and_multi_echo_denoising.ipynb'
NOTEBOOK_RELATIVE_PATH = '07_00_mri_preprocessing_and_multi_echo_denoising.ipynb'
WORKING_DIRECTORY = Path.cwd().resolve()
NOTEBOOK_CANDIDATES = [Path(os.environ['A8_NOTEBOOK_PATH']).expanduser()] if 'A8_NOTEBOOK_PATH' in os.environ else [WORKING_DIRECTORY / NOTEBOOK_NAME, WORKING_DIRECTORY / NOTEBOOK_RELATIVE_PATH]
NOTEBOOK_PATH = next((p.resolve() for p in NOTEBOOK_CANDIDATES if p.is_file()), None)
if NOTEBOOK_PATH is None:
    raise FileNotFoundError('Set A8_NOTEBOOK_PATH to this notebook, or run with its containing directory as the working directory')
ANALYSIS_ROOT = NOTEBOOK_PATH.parent


In [ ]:
from pathlib import Path as u11_Path
import sys as u11_sys
from IPython.display import display as u11_display, Image as u11_Image
from pathlib import Path as u11_Path
import os as u11_os, json as u11_json, re as u11_re, subprocess as u11_subprocess
import numpy as u11_np
import pandas as u11_pd
import nibabel as u11_nib
u11_preprocess_TE_SECONDS = (0.0117, 0.0319, 0.0521, 0.0723)


In [ ]:
def u11_preprocess_command_program(name, version):
    program = u11_os.environ.get('A8_' + name.upper() + '_EXECUTABLE', name)
    result = u11_subprocess.run([program, '--version'], check=True, capture_output=True, text=True)
    if not u11_re.search('(?<![0-9.])' + u11_re.escape(version) + '(?![0-9.])', result.stdout + ' ' + result.stderr):
        raise RuntimeError(name + ' must match manuscript version ' + version)
    return program


In [ ]:
def u11_preprocess_directory(value):
    path = u11_Path(str(value)).expanduser()
    return path.resolve() if path.is_absolute() else (u11_DATA / path).resolve()


In [ ]:
def u11_preprocess_run_fmriprep():
    cfg = u11_table('MRI_preprocessing.csv', ['bids_dir', 'fmriprep_dir', 'work_dir', 'fs_license'])
    if len(cfg) != 1:
        raise ValueError('MRI_preprocessing.csv must contain one dataset configuration')
    row = cfg.iloc[0]
    (bids, output, work) = [u11_preprocess_directory(row[k]) for k in ['bids_dir', 'fmriprep_dir', 'work_dir']]
    if not bids.is_dir() or not (bids / 'dataset_description.json').is_file():
        raise FileNotFoundError('Supply an existing raw BIDS dataset')
    if bids == output or bids == work or output == work:
        raise ValueError('Raw BIDS, derivative and work directories must differ')
    labels = [x.removeprefix('sub-') for x in u11_participants().subject_id]
    for label in labels:
        if not u11_re.fullmatch('[A-Za-z0-9]+', label) or not (bids / ('sub-' + label)).is_dir():
            raise ValueError('BIDS participant missing: ' + label)
    executable = u11_preprocess_command_program('fmriprep', '24.1.1')
    output.mkdir(parents=True, exist_ok=True)
    work.mkdir(parents=True, exist_ok=True)
    command = [executable, str(bids), str(output), 'participant', '--participant-label', *labels, '--output-spaces', 'MNI152NLin2009cAsym:res-2', 'T1w', '--me-output-echos', '--fs-license-file', str(u11_file_ref(row.fs_license)), '--fd-spike-threshold', '0.5', '--dvars-spike-threshold', '1.5', '--random-seed', '20261022', '--notrack', '-w', str(work)]
    u11_subprocess.run(command, check=True)
    u11_provenance('fmriprep_preprocessing', ['Participants.csv', 'MRI_preprocessing.csv'])
    return output


In [ ]:
def u11_preprocess_checked_image(path, shape=None, affine=None):
    im = u11_nib.load(path)
    if shape is not None and im.shape != shape:
        raise ValueError('Image dimensions mismatch: ' + str(path))
    if affine is not None and (not u11_np.allclose(im.affine, affine)):
        raise ValueError('Image grid mismatch: ' + str(path))
    return im


In [ ]:
def u11_preprocess_warp_image(source, reference, transforms, output, time_series=False):
    command = [u11_os.environ.get('A8_ANTS_EXECUTABLE', 'antsApplyTransforms'), '-d', '3', '-e', '3' if time_series else '0', '-i', str(source), '-r', str(reference), '-o', str(output), '-n', 'LanczosWindowedSinc' if time_series else 'GenericLabel', '--float', '1']
    for path in transforms:
        command.extend(['-t', str(path)])
    u11_subprocess.run(command, check=True)


In [ ]:
def u11_preprocess_run_tedana():
    columns = ['subject_id', 'task', 'run', 'echo_1', 'echo_2', 'echo_3', 'echo_4', 'native_mask', 'confounds', 'metadata', 'events', 'reference', 'transforms']
    d = u11_table('MultiEcho_runs.csv', columns, ['subject_id', 'task', 'run'])
    if set(d.subject_id) != set(u11_participants().subject_id) or not d.task.isin(['stroop', 'gonogo', 'rest']).all():
        raise ValueError('Multi-echo input cohort/task mismatch')
    d['run'] = d.run.fillna('single').astype(str).str.replace('\\.0$', '', regex=True).str.lstrip('0').replace({'': 'single'})
    if d.duplicated(['subject_id', 'task', 'run']).any():
        raise ValueError('Duplicate scan after run-label normalization')
    for (sid, g) in d.groupby('subject_id'):
        counts = g.groupby('task').size().to_dict()
        if counts != {'gonogo': 2, 'rest': 1, 'stroop': 1} or set(g.loc[g.task.eq('gonogo'), 'run']) != {'1', '2'}:
            raise ValueError('Expected one Stroop, two Go/No-Go and one rest scan: ' + sid)
    executable = u11_preprocess_command_program('tedana', '26.0.3')
    records = []
    for row in d.itertuples():
        echoes = [u11_file_ref(getattr(row, 'echo_' + str(i))) for i in range(1, 5)]
        first = u11_preprocess_checked_image(echoes[0])
        if len(first.shape) != 4:
            raise ValueError('Echo input must be a minimally preprocessed 4D image')
        for path in echoes[1:]:
            u11_preprocess_checked_image(path, first.shape, first.affine)
        native_mask = u11_file_ref(row.native_mask)
        mask = u11_preprocess_checked_image(native_mask, first.shape[:3], first.affine)
        mask_values = u11_np.asarray(mask.dataobj)
        if not u11_np.isin(mask_values, [0, 1]).all() or not mask_values.any():
            raise ValueError('Native mask must be nonempty and binary')
        reference = u11_file_ref(row.reference)
        ref = u11_preprocess_checked_image(reference)
        if len(ref.shape) != 3 or not u11_np.allclose(u11_nib.affines.voxel_sizes(ref.affine), 2):
            raise ValueError('Reference must use the manuscript 2-mm MNI152NLin2009cAsym grid')
        transform_names = u11_json.loads(row.transforms)
        if not isinstance(transform_names, list) or not transform_names or (not all((isinstance(x, str) for x in transform_names))):
            raise ValueError('transforms must list native-to-standard transforms in ANTs CLI order')
        transforms = [u11_file_ref(x) for x in transform_names]
        confounds = u11_file_ref(row.confounds)
        cf = u11_pd.read_csv(confounds, sep=',' if confounds.suffix.lower() == '.csv' else '\t', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
        if len(cf) != first.shape[3]:
            raise ValueError('Confounds and echo volumes differ')
        metadata = u11_file_ref(row.metadata)
        meta = u11_json.loads(metadata.read_text())
        if not u11_np.isclose(float(meta['RepetitionTime']), 2):
            raise ValueError('RepetitionTime differs from manuscript 2 s')
        if 'EchoTimes' in meta and (not u11_np.allclose(meta['EchoTimes'], u11_preprocess_TE_SECONDS)):
            raise ValueError('Echo times differ from the four manuscript echo times')
        events = '' if row.task == 'rest' else str(u11_file_ref(row.events))
        dest = u11_preprocess_OUT / 'preprocessing' / (str(row.subject_id) + '_' + row.task + '_run-' + str(row.run))
        dest.mkdir(parents=True, exist_ok=True)
        native = dest / 'tedana'
        command = [executable, '-d', *[str(p) for p in echoes], '-e', *[str(te) for te in u11_preprocess_TE_SECONDS], '--mask', str(native_mask), '--out-dir', str(native), '--convention', 'bids', '--seed', '20261022', '--overwrite']
        u11_subprocess.run(command, check=True)
        denoised = native / 'desc-denoised_bold.nii.gz'
        u11_preprocess_checked_image(denoised, first.shape, first.affine)
        bold = dest / 'space-MNI152NLin2009cAsym_res-2_desc-denoised_bold.nii.gz'
        target_mask = dest / 'space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz'
        u11_preprocess_warp_image(denoised, reference, transforms, bold, time_series=True)
        u11_preprocess_warp_image(native_mask, reference, transforms, target_mask)
        u11_preprocess_checked_image(bold, ref.shape + (first.shape[3],), ref.affine)
        mapped_mask = u11_preprocess_checked_image(target_mask, ref.shape, ref.affine)
        if not u11_np.isin(u11_np.asarray(mapped_mask.dataobj), [0, 1]).all() or not u11_np.asarray(mapped_mask.dataobj).any():
            raise ValueError('Invalid transformed brain mask')
        target_metadata = dest / 'acquisition.json'
        meta.update(SpatialReference='MNI152NLin2009cAsym', EchoTimes=list(u11_preprocess_TE_SECONDS), PreprocessingVersions={'fMRIPrep': '24.1.1', 'tedana': '26.0.3'})
        target_metadata.write_text(u11_json.dumps(meta, indent=2))
        records.append(dict(subject_id=row.subject_id, task=row.task, run=row.run, bold=str(bold), mask=str(target_mask), confounds=str(confounds), metadata=str(target_metadata), events=events))
    result = u11_pd.DataFrame(records)
    target = u11_preprocess_OUT / 'tables/MRI_manifest.csv'
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix('.pending.csv')
    result.to_csv(temp, index=False)
    temp.replace(target)
    u11_provenance('tedana_and_standard_space', ['Participants.csv', 'MultiEcho_runs.csv'])
    return result


In [ ]:
from pathlib import Path as u15_Path
import os as u15_os, hashlib as u15_hashlib, json as u15_json
from importlib import metadata as u15_metadata
import platform as u15_platform, sys as u15_sys
import numpy as u15_np
import pandas as u15_pd
u15_ROOT = ANALYSIS_ROOT
u15_DATA = u15_Path(u15_os.environ.get('A8_DATA_DIR', NOTEBOOK_PATH.parent)).expanduser().resolve()
u15_RESULTS = u15_Path(u15_os.environ.get('A8_OUTPUT_DIR', u15_ROOT / 'results')).expanduser().resolve()
u15_EXPECTED_N = int(u15_os.environ.get('A8_EXPECTED_N', '106'))
u15_PARAMS = ['stroop_a', 'stroop_v_congruent', 'stroop_v_neutral', 'stroop_v_incongruent', 'stroop_t', 'gonogo_a', 'gonogo_v_go', 'gonogo_v_nogo', 'gonogo_t', 'gonogo_z']
u15_INPUT_FOLDERS = {'MRI_manifest.csv': '7.task_fmri', 'Participants.csv': '1.behavior', 'ROI_masks.csv': '7.task_fmri', 'HDDM_posterior_draws.csv': '2.hddm_fitting', 'HDDM_parameters.csv': '2.hddm_fitting', 'Behavior_data.csv': '1.behavior', 'HDDM_data.csv': '2.hddm_fitting'}
u15_INPUT_FOLDERS.update({'MRI_preprocessing.csv': '7.task_fmri', 'MultiEcho_runs.csv': '7.task_fmri'})
u15_INPUT_FILES = {name: name for name in u15_INPUT_FOLDERS}
u15_INPUT_COLUMNS = {'Participants.csv': ('subject_id', 'age', 'sex', 'education'), 'Behavior_data.csv': ('subject_id', 'task', 'run', 'trial_id', 'condition', 'rt_s', 'response', 'correct'), 'HDDM_data.csv': ('subj_idx', 'task', 'run', 'trial_id', 'condition', 'stimulus', 'response', 'rt'), 'MRI_manifest.csv': ('subject_id', 'task', 'run', 'bold', 'mask', 'confounds', 'metadata', 'events'), 'ROI_masks.csv': ('roi_id', 'path', 'source_doi', 'independent_of_sample'), 'HDDM_parameters.csv': ('subject_id', *u15_PARAMS), 'HDDM_posterior_draws.csv': ('task', 'chain', 'draw', 'subject_id', 'parameter', 'value')}


In [ ]:
def u15_input_path(name):
    filename = u15_INPUT_FILES.get(name, name)
    p = u15_DATA / filename
    if not p.is_file() and 'A8_DATA_DIR' not in u15_os.environ:
        p = u15_ROOT / u15_INPUT_FOLDERS.get(name, '7.task_fmri') / filename
    if name == 'MRI_manifest.csv' and (u15_os.environ.get('A8_MRI_INPUT_MODE') == 'generated' or not p.is_file()):
        p = u15_RESULTS / '7.task_fmri/tables/MRI_manifest.csv'
    fitted = (u15_RESULTS / '2.hddm_fitting/fit_state.json').is_file() and u15_os.environ.get('A8_HDDM_INPUT_MODE', 'auto') != 'external'
    if name in {'HDDM_parameters.csv', 'HDDM_posterior_draws.csv'} and (fitted or not p.is_file()):
        p = u15_RESULTS / '2.hddm_fitting/tables' / name
    if not p.is_file():
        raise FileNotFoundError(f'Required observed input missing: {p}. Supply the required real input file; no example values are substituted.')
    return p


In [ ]:
def u15_table(name, columns=(), unique=None):
    p = u15_input_path(name)
    d = u15_pd.read_csv(p, dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    if d.empty:
        raise ValueError(f'{p.name}: no observed records; a header-only template is not data')
    missing = (set(columns) | set(u15_INPUT_COLUMNS.get(name, ()))) - set(d)
    if missing:
        raise ValueError(f'{p.name}: missing columns {sorted(missing)}')
    if unique and (d[list(unique)].isna().any().any() or d.duplicated(list(unique)).any()):
        raise ValueError(f'{p.name}: missing/duplicate key {unique}')
    for c in ('subject_id', 'subj_idx', 'trial_id'):
        if c in d and (d[c].isna().any() or d[c].str.strip().eq('').any() or d[c].ne(d[c].str.strip()).any()):
            raise ValueError(f'{p.name}: blank or whitespace-padded identifier in {c}')
    for c in ('data_status', 'source_status'):
        if c in d and d[c].astype(str).str.contains('synthetic|simulation|not empirical', case=False, regex=True).any():
            raise ValueError(f'{p.name}: simulated rows cannot enter this workflow')
    return d


In [ ]:
def u15_participants():
    d = u15_table('Participants.csv', ['subject_id', 'age', 'sex', 'education'], ['subject_id'])
    if len(d) != u15_EXPECTED_N:
        raise ValueError(f'Expected {u15_EXPECTED_N} participants, found {len(d)}; configure A8_EXPECTED_N deliberately.')
    for c in ['age', 'sex', 'education']:
        d[c] = u15_pd.to_numeric(d[c], errors='raise')
        if not u15_np.isfinite(d[c]).all():
            raise ValueError(f'Nonfinite {c}')
    if not d.sex.isin([0, 1]).all():
        raise ValueError('sex must use the documented 0/1 coding')
    return d.sort_values('subject_id').reset_index(drop=True)


In [ ]:
def u15_file_ref(value):
    if u15_pd.isna(value) or not str(value).strip():
        raise ValueError('Empty input path')
    p = u15_Path(str(value)).expanduser()
    p = p if p.is_absolute() else u15_DATA / p
    if not p.is_file():
        raise FileNotFoundError(p)
    return p.resolve()


In [ ]:
def u15_source_signature():
    document = u15_json.loads(NOTEBOOK_PATH.read_text())
    source = '\n'.join((''.join(cell['source']) for cell in document['cells'] if cell['cell_type'] == 'code'))
    return u15_hashlib.sha256(source.encode()).hexdigest()


In [ ]:
def u15_runtime_versions():
    versions = {'python': u15_sys.version, 'platform': u15_platform.platform()}
    for name in ['numpy', 'scipy', 'pandas', 'statsmodels', 'scikit-learn', 'nilearn', 'nibabel', 'arviz', 'hddm', 'kabuki', 'pymc', 'matplotlib', 'openpyxl', 'antspyx', 'templateflow']:
        try:
            versions[name] = u15_metadata.version(name)
        except u15_metadata.PackageNotFoundError:
            versions[name] = None
    return versions


In [ ]:
def u15_atomic_record(path, record):
    path = u15_Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.pending')
    temp.write_text(u15_json.dumps(record, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)


In [ ]:
def u15_provenance(stage, names, settings=None):
    records = []
    for name in names:
        path = u15_input_path(name)
        records.append({'file': str(path), 'sha256': u15_hashlib.sha256(path.read_bytes()).hexdigest()})
    record = dict(stage=stage, input_files=records, code_sha256=u15_source_signature(), runtime=u15_runtime_versions(), settings=settings or {})
    u15_atomic_record(u15_RESULTS / (stage + '_inputs.json'), record)
    return record


In [ ]:
from pathlib import Path as u17__A8Path
from pathlib import Path as u17_Path
import sys as u17_sys
import hashlib as u17_hashlib
import json as u17_json
import os as u17_os
for u17__key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    u17_os.environ.setdefault(u17__key, '1')
import nibabel as u17_nib
import numpy as u17_np
import pandas as u17_pd
import nilearn as u17_nilearn
if u17_nilearn.__version__ != '0.13.1':
    raise RuntimeError('Use Nilearn 0.13.1, as specified in the manuscript')
from scipy import stats as u17_stats
from statsmodels.stats.multitest import multipletests as u17_multipletests
import sys as u17_sys
from pathlib import Path as u17_Path
u17_PROJECT = u15_ROOT
u17_RESULTS = u15_RESULTS
u17_DATA = u15_DATA
u17_EXPECTED_N = u15_EXPECTED_N
u17_participants = u15_participants
u17_source_signature = u15_source_signature
u17_MRI = u17_RESULTS / '3.mri_analysis'
u17_TASK = u17_MRI / '2.task_fmri'
u17_REST = u17_MRI / '1.resting_state'
u17_VERSION = 'observed_inputs_v1'
u17_LEGACY = u17_DATA / 'ale_masks'
u17_NEW = u17_TASK / 'other_outputs'
u17_REST_NEW = u17_REST / 'other_outputs'
u17_ROI_CONFIG = u17_MRI / 'ROI_definition_ALE_3D_clusters.json'
u17_ROI_VARIANTS = ('ALE_3D',)
u17_ROIS = ['shared_SMA', 'shared_rAI_FO', 'shared_lAI', 'stroop_lIPS', 'gonogo_rIFG_PMC']
u17_CONTRAST = {'stroop': 'incongruent_minus_congruent', 'gonogo': 'nogo_correct_minus_go_correct'}
u17_N_PERM = 50000
u17_SEED = 20261017


In [ ]:
def u17_subjects():
    return u17_participants().subject_id.tolist()


In [ ]:
def u17_sha(path):
    h = u17_hashlib.sha256()
    with u17_Path(path).open('rb') as f:
        for block in iter(lambda : f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


In [ ]:
def u17_fingerprint(paths, extra=None):
    info = [(str(u17_Path(p).resolve()), u17_Path(p).stat().st_size, u17_Path(p).stat().st_mtime_ns) for p in paths]
    return u17_hashlib.sha256(u17_json.dumps([info, extra, u17_source_signature()], sort_keys=True).encode()).hexdigest()


In [ ]:
def u17_atomic_json(path, data):
    path = u17_Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')

    def native(value):
        if isinstance(value, u17_np.generic):
            return value.item()
        if isinstance(value, u17_np.ndarray):
            return value.tolist()
        raise TypeError(f'Unsupported metadata type: {type(value).__name__}')
    tmp.write_text(u17_json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False, default=native))
    tmp.replace(path)


In [ ]:
def u17_save_table(frame, root, name):
    if '10_activation_tests' in name or name.startswith('functional_connectivity_10_edges_'):
        frame = frame.drop(columns=['p_flip_positive'], errors='ignore')
    if name.startswith('functional_connectivity_10_edges_'):
        frame = frame.drop(columns=['t', 'df', 'p_t'], errors='ignore')
    if 'pattern_similarity_' in name:
        frame = frame.drop(columns=['t', 'df', 'p_t'], errors='ignore')
    dest = root / 'tables'
    dest.mkdir(parents=True, exist_ok=True)
    path = dest / f'{name}.xlsx'
    frame.to_excel(path, index=False)
    from openpyxl import load_workbook as load_workbook
    from openpyxl.styles import Font as Font, Border as Border, Side as Side, Alignment as Alignment
    from openpyxl.utils import get_column_letter as get_column_letter
    wb = load_workbook(path)
    ws = wb.active
    rule = Side(style='thin', color='333333')
    ws.sheet_view.showGridLines = False
    ws.freeze_panes = 'A2'
    for (col, key) in enumerate(frame.columns, 1):
        letter = get_column_letter(col)
        ws.column_dimensions[letter].width = min(38, max(13, len(str(key)) + 2))
        cell = ws.cell(1, col)
        cell.font = Font(name='Arial', size=10, bold=True)
        cell.alignment = Alignment(wrap_text=True, vertical='center')
        cell.border = Border(top=rule, bottom=rule)
        for row in range(2, ws.max_row + 1):
            c = ws.cell(row, col)
            c.font = Font(name='Arial', size=10)
            if isinstance(c.value, str):
                c.alignment = Alignment(wrap_text=True, vertical='top')
                visual_length = sum((2 if ord(char) > 255 else 1 for char in c.value))
                lines = max(1, int(u17_np.ceil(visual_length / ws.column_dimensions[letter].width)))
                ws.row_dimensions[row].height = max(ws.row_dimensions[row].height or 15, min(150, lines * 15))
            if isinstance(c.value, float):
                c.number_format = '0.000E+00' if str(key).startswith(('p', 'q')) else '0.000'
            if row == ws.max_row:
                c.border = Border(bottom=rule)
    ws.row_dimensions[1].height = 32
    wb.save(path)


In [ ]:
def u17_fdr(p):
    p = u17_np.asarray(p, float)
    out = u17_np.full(p.shape, u17_np.nan)
    valid = u17_np.isfinite(p)
    if valid.any():
        out[valid] = u17_multipletests(p[valid], method='fdr_bh')[1]
    return out


In [ ]:
def u17_mean_stats(x, seed=u17_SEED, alternative='two-sided'):
    if alternative not in ('two-sided', 'greater'):
        raise ValueError('Unsupported alternative')
    key = 'p_flip' if alternative == 'two-sided' else 'p_flip_positive'
    x = u17_np.asarray(x, float)
    x = x[u17_np.isfinite(x)]
    n = len(x)
    if n < 3:
        return dict(N=n, mean=u17_np.nan, SD=u17_np.nan, t=u17_np.nan, df=n - 1, CI95_low=u17_np.nan, CI95_high=u17_np.nan, **{key: u17_np.nan})
    mu = x.mean()
    rng = u17_np.random.default_rng(seed)
    exceed = 0
    for start in range(0, u17_N_PERM, 1000):
        signs = rng.choice((-1.0, 1.0), size=(min(1000, u17_N_PERM - start), n))
        perm = signs @ x / n
        exceed += u17_np.count_nonzero(abs(perm) >= abs(mu) - 1e-14) if alternative == 'two-sided' else u17_np.count_nonzero(perm >= mu - 1e-14)
    sem = u17_stats.sem(x)
    ci = u17_stats.t.interval(0.95, n - 1, loc=mu, scale=sem)
    return dict(N=n, mean=float(mu), SD=float(x.std(ddof=1)), t=float(mu / sem), df=n - 1, CI95_low=float(ci[0]), CI95_high=float(ci[1]), **{key: (exceed + 1) / (u17_N_PERM + 1)})


In [ ]:
def u17_first_path(sid, task, stat):
    name = f'{sid}_task-{task}_contrast-{u17_CONTRAST[task]}_stat-{stat}.nii.gz'
    fresh = u17_NEW / name
    if not u17_new_first_complete(sid, task):
        raise FileNotFoundError(f'Current first-level GLM is incomplete; fallback to previous input results is prohibited: {sid}/{task}')
    return fresh


In [ ]:
def u17_new_first_complete(sid, task):
    suffix = 'stroop' if task == 'stroop' else 'gonogo_duration500ms'
    qc = u17_NEW / f'{sid}_task-{suffix}_first_level_qc.csv'
    maps = [u17_NEW / f'{sid}_task-{task}_contrast-{u17_CONTRAST[task]}_stat-{stat}.nii.gz' for stat in ('effect', 'variance', 'z')]
    maps.append(u17_NEW / f'{sid}_task-{task}_mask-analysis.nii.gz')
    return qc.exists() and all((p.exists() for p in maps))


In [ ]:
def u17_qc_path(sid, task):
    suffix = 'stroop' if task == 'stroop' else 'gonogo_duration500ms'
    name = f'{sid}_task-{suffix}_first_level_qc.csv'
    fresh = u17_NEW / name
    if not u17_new_first_complete(sid, task):
        raise FileNotFoundError(f'Current first-level GLM is incomplete: {sid}/{task}')
    return fresh


In [ ]:
def u17_t_map(sid, task):
    e = u17_nib.load(u17_first_path(sid, task, 'effect'))
    v = u17_nib.load(u17_first_path(sid, task, 'variance'))
    if e.shape != v.shape or not u17_np.allclose(e.affine, v.affine):
        raise ValueError('Effect and variance image spaces do not match')
    (ed, vd) = (u17_np.asarray(e.dataobj, float), u17_np.asarray(v.dataobj, float))
    valid = u17_np.isfinite(ed) & u17_np.isfinite(vd) & (vd > 0)
    t = u17_np.divide(ed, u17_np.sqrt(u17_np.maximum(vd, 0)), out=u17_np.zeros_like(ed), where=valid)
    qc = u17_pd.read_csv(u17_qc_path(sid, task), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    dof = int((qc.n_timepoints - qc.design_rank).sum())
    return (t, valid, e, dof)


In [ ]:
def u17_roi_masks(definition='ALE_3D'):
    if definition != 'ALE_3D':
        raise ValueError('The specified analysis uses independent three-dimensional ALE cluster masks only')
    cfg = u17_json.loads(u17_ROI_CONFIG.read_text())
    manifest = u17_NEW / 'definition_and_provenance.json'
    if not manifest.exists():
        raise FileNotFoundError('Provide and validate independent ALE ROI inputs, or run the ROI definition step in this notebook')
    meta = u17_json.loads(manifest.read_text())
    if meta['config_sha256'] != u17_sha(u17_ROI_CONFIG):
        raise ValueError('ROI definition changed; regenerate ROI masks')
    ref = u17_nib.load(u17_first_path(u17_subjects()[0], 'stroop', 'effect'))
    masks = {}
    for roi in u17_ROIS:
        path = u17_NEW / f'{roi}_ALE_3D.nii.gz'
        if u17_sha(path) != meta['masks'][roi]['sha256']:
            raise ValueError('ALE mask validation failed')
        img = u17_nib.load(path)
        data = u17_np.asarray(img.dataobj)
        if img.shape != ref.shape or not u17_np.allclose(img.affine, ref.affine):
            raise ValueError('ALE ROI and subject statistic image spaces do not match')
        if not u17_np.isin(data, [0, 1]).all() or not data.any():
            raise ValueError('ALE mask must be nonempty and binary')
        masks[roi] = data > 0
    return (masks, ref)


In [ ]:
u11_ROOT = u15_ROOT
u11_DATA = u15_DATA
u11_RESULTS = u15_RESULTS
u11_DATA = u15_DATA
u11_RESULTS = u15_RESULTS
u11_table = u15_table
u11_participants = u15_participants
u11_file_ref = u15_file_ref
u11_provenance = u15_provenance
u11_preprocess_OUT = u11_RESULTS / '7.task_fmri'


In [ ]:
def a8_display_three_line(value, *args, **kwargs):
    import pandas as pd
    from IPython.display import display, HTML
    if isinstance(value, pd.DataFrame):
        html = value.to_html(index=False, border=0, escape=True)
        html = html.replace('<table ', '<table style="border-collapse:collapse;border-top:2px solid black;border-bottom:2px solid black;" ', 1)
        html = html.replace('<th>', '<th style="border:0;border-bottom:1px solid black;padding:7px 16px;text-align:center;">')
        html = html.replace('<td>', '<td style="border:0;padding:7px 16px;text-align:center;">')
        return display(HTML(html), *args, **kwargs)
    return display(value, *args, **kwargs)
u11_display = a8_display_three_line


In [ ]:
u11__A8Path = u17__A8Path
u11_Path = u17_Path
u11_sys = u17_sys
u11_hashlib = u17_hashlib
u11_json = u17_json
u11_os = u17_os
u11_nib = u17_nib
u11_np = u17_np
u11_pd = u17_pd
u11_nilearn = u17_nilearn
u11_stats = u17_stats
u11_multipletests = u17_multipletests
u11_PROJECT = u17_PROJECT
u11_RESULTS = u17_RESULTS
u11_DATA = u17_DATA
u11_EXPECTED_N = u17_EXPECTED_N
u11_participants = u17_participants
u11_source_signature = u17_source_signature
u11_MRI = u17_MRI
u11_TASK = u17_TASK
u11_REST = u17_REST
u11_VERSION = u17_VERSION
u11_LEGACY = u17_LEGACY
u11_NEW = u17_NEW
u11_REST_NEW = u17_REST_NEW
u11_ROI_CONFIG = u17_ROI_CONFIG
u11_ROI_VARIANTS = u17_ROI_VARIANTS
u11_ROIS = u17_ROIS
u11_CONTRAST = u17_CONTRAST
u11_N_PERM = u17_N_PERM
u11_SEED = u17_SEED
u11_subjects = u17_subjects
u11_sha = u17_sha
u11_fingerprint = u17_fingerprint
u11_atomic_json = u17_atomic_json
u11_save_table = u17_save_table
u11_fdr = u17_fdr
u11_mean_stats = u17_mean_stats
u11_first_path = u17_first_path
u11_new_first_complete = u17_new_first_complete
u11_qc_path = u17_qc_path
u11_t_map = u17_t_map
u11_roi_masks = u17_roi_masks


In [ ]:
u11_preprocess_run_fmriprep()


In [ ]:
u11_preprocessed_runs = u11_preprocess_run_tedana()
u11_display(u11_preprocessed_runs)
